In [4]:
!pip uninstall -y massi
!pip cache purge

Found existing installation: massi 0.1.0
Uninstalling massi-0.1.0:
  Successfully uninstalled massi-0.1.0
Files removed: 6 (1.2 MB)


In [5]:
!cd .. && pip install -v . --group=dev --group=observability --group=sandbox

Using pip 25.3 from /opt/conda/lib/python3.13/site-packages/pip (python 3.13)
Processing /home/jovyan/work
  Running command installing build dependencies
  Using pip 25.3 from /opt/conda/lib/python3.13/site-packages/pip (python 3.13)
    Obtaining dependency information for setuptools>=40.8.0 from https://files.pythonhosted.org/packages/9d/76/f789f7a86709c6b087c5a2f52f911838cad707cc613162401badc665acfe/setuptools-82.0.1-py3-none-any.whl.metadata
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 0.0/1.0 MB ? eta -:--:--
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 10.8 MB/s  0:00:00
  Installing build dependencies ... done
  Running command Getting requirements to build wheel
  running egg_info
  writing massi.egg-info/PKG-INFO
  writing dependency_links to massi.egg-info/dependency_links.txt
  writing requirements to massi.egg-info/requires.txt
  writing top-level names to massi.egg-info/top_level.txt
  reading manifest file 'massi.egg-info/SOURCES.txt'
  writing manifest file

In [6]:
import massi.agents.tools as mts

In [11]:
df = mts.safe_get_csv_details(
    "../_data/gbif_data_archive/working_copy/data/001-2fee34a0-a302-3c12-8916-bd915a1c0e50-download-data/occurrence.txt"
)

Error tokenizing data. C error: Expected 17 fields in line 12, saw 18



In [12]:
import logging
import os
import pathlib as pl
from abc import ABC, abstractmethod
from dataclasses import dataclass
from typing import Any, Iterator
import warnings

import pandas as pd
import xmlschema as xs

from massi.agents.team._dto import CSVDetails

In [42]:

def get_csv_details(csv: pl.Path, separator: str | None = None) -> CSVDetails:
    head_size: int = 5
    with open(csv) as fp:
        raw_head = "\n".join([next(fp) for _ in range(head_size)])
    df = pd.read_csv(csv, sep=separator, on_bad_lines='warn')
    details: CSVDetails = {
        "column_count": df.shape[1],
        "row_count": df.shape[0],
        "head_size": head_size,
        "raw_head": raw_head,
        "dataframe_head": df.head(head_size).to_csv(None, sep=",", index=False),
        "descriptive_stats": df.describe(include="all").to_csv(
            None, sep=",", index=False
        ),
    }
    del df
    return details


def __safe_get_csv_details(
    csv: pl.Path, separator: str | None = None
) -> CSVDetails | None:
    try:
        return get_csv_details(csv, separator=separator)
    except Exception as e:
        print(e)
        # logger.warning(e)
        warnings.warn(f"Could not read the CSV: {csv=} -->\n{e=}")
        return None


def safe_get_csv_details(csv: pl.Path) -> CSVDetails | None:
    for separator in [None, "\t", ","]:
        print(f"Trying {separator=}")
        details = __safe_get_csv_details(csv, separator=separator)
        if details is not None:
            return details
    warnings.warn(f"Could not read the CSV contents: {csv=}...")
    return None

In [43]:
data = safe_get_csv_details(
    "../_data/gbif_data_archive/working_copy/data/001-2fee34a0-a302-3c12-8916-bd915a1c0e50-download-data/occurrence.txt"
)

Trying separator=None
object of type 'NoneType' has no len()
Trying separator='\t'


/tmp/ipykernel_43186/3245544550.py:28: UserWarning: Could not read the CSV: csv='../_data/gbif_data_archive/working_copy/data/001-2fee34a0-a302-3c12-8916-bd915a1c0e50-download-data/occurrence.txt' -->
e=TypeError("object of type 'NoneType' has no len()")
  warnings.warn(f"Could not read the CSV: {csv=} -->\n{e=}")
/tmp/ipykernel_43186/3245544550.py:5: DtypeWarning: Columns (0: minimumElevationInMeters, 1: maximumElevationInMeters) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(csv, sep=separator, on_bad_lines='warn')


In [44]:
data

{'column_count': 38,
 'row_count': 34588,
 'head_size': 5,
 'raw_head': 'id\tlicense\tbasisOfRecord\tdynamicProperties\toccurrenceID\tgeoreferenceVerificationStatus\toccurrenceStatus\tassociatedReferences\teventDate\tyear\tmonth\tday\tcountry\tcountryCode\tstateProvince\tlocality\tverbatimLocality\tminimumElevationInMeters\tmaximumElevationInMeters\tdecimalLatitude\tdecimalLongitude\tgeodeticDatum\tcoordinateUncertaintyInMeters\tcoordinatePrecision\tgeoreferencedBy\tverbatimIdentification\tidentifiedBy\tscientificName\tnameAccordingTo\tkingdom\tphylum\tclass\torder\tfamily\tgenus\tspecificEpithet\ttaxonRank\tscientificNameAuthorship\n\n65440495-15f3-5e13-a4f5-5ab1eafe8f7e-a\tCC-BY-4.0\tHumanObservation\t{"airTemperature_C":null,"annualPrecipitation_mmYr":null,"habitatPressure":"Near-natural","forestTypeDetailed":"mountain Fagus-Carpinus forest","consolidatedForestCategory":null,"microhabitat":"bark of Pinus pallasiana (scorched trunk)","substrateCategory":"Corticolous substrates","pH":

In [53]:
csv_path = (
    "../_data/gbif_data_archive/working_copy/"
    + "data/001-2fee34a0-a302-3c12-8916-bd915a1c0e50-download-data/"
    + "occurrence.txt"
)

In [55]:
df = pd.read_csv(csv_path, sep="\t", on_bad_lines='warn')

/tmp/ipykernel_43186/57373013.py:1: DtypeWarning: Columns (0: minimumElevationInMeters, 1: maximumElevationInMeters) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(csv_path, sep="\t", on_bad_lines='warn')


In [56]:
df

,id,license,basisOfRecord,dynamicProperties,occurrenceID,georeferenceVerificationStatus,occurrenceStatus,associatedReferences,eventDate,year,...,nameAccordingTo,kingdom,phylum,class,order,family,genus,specificEpithet,taxonRank,scientificNameAuthorship
0,65440495-15f3-5e13-a4f5-5ab1eafe8f7e-a,CC-BY-4.0,HumanObservation,"{""airTemperature_C"":null,""annualPrecipitation_...",65440495-15f3-5e13-a4f5-5ab1eafe8f7e-a,verified,present,"Leontyev, D. V.; McHugh, R.; Fefelov, K. A.; K...",2011-02,2011,...,Eumycetozoa.com,Protozoa,Mycetozoa,Myxomycetes,Stemonitidales,Stemonitidaceae,Amaurochaete,atra,species,"(Alb. & Schwein.) Rostaf., 1874"
1,65440495-15f3-5e13-a4f5-5ab1eafe8f7e-b,CC-BY-4.0,HumanObservation,"{""airTemperature_C"":null,""annualPrecipitation_...",65440495-15f3-5e13-a4f5-5ab1eafe8f7e-b,verified,present,"Leontyev, D. V.; McHugh, R.; Fefelov, K. A.; K...",2011-02,2011,...,Eumycetozoa.com,Protozoa,Mycetozoa,Myxomycetes,Stemonitidales,Stemonitidaceae,Amaurochaete,atra,species,"(Alb. & Schwein.) Rostaf., 1874"
2,65440495-15f3-5e13-a4f5-5ab1eafe8f7e-c,CC-BY-4.0,HumanObservation,NaN,65440495-15f3-5e13-a4f5-5ab1eafe8f7e-c,verified,present,"Leontyev, D. V.; McHugh, R.; Fefelov, K. A.; K...",2011-02,2011,...,Eumycetozoa.com,Protozoa,Mycetozoa,Myxomycetes,Stemonitidales,Stemonitidaceae,Amaurochaete,atra,species,"(Alb. & Schwein.) Rostaf., 1874"
3,65440495-15f3-5e13-a4f5-5ab1eafe8f7e,CC-BY-4.0,HumanObservation,NaN,65440495-15f3-5e13-a4f5-5ab1eafe8f7e,verified,present,"Leontyev, D. V.; McHugh, R.; Fefelov, K. A.; K...",2011-02,2011,...,Eumycetozoa.com,Protozoa,Mycetozoa,Myxomycetes,Stemonitidales,Stemonitidaceae,Amaurochaete,atra,species,"(Alb. & Schwein.) Rostaf., 1874"
4,96a96498-011c-507f-95d5-65d99bd3d539,CC-BY-4.0,HumanObservation,NaN,96a96498-011c-507f-95d5-65d99bd3d539,verified,present,"Meylan, C. (1924). Recherches sur les Myxomycè...",1924,1924,...,Eumycetozoa.com,Protozoa,Mycetozoa,Myxomycetes,Stemonitidales,Stemonitidaceae,Amaurochaete,atra,species,"(Alb. & Schwein.) Rostaf., 1874"
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
34583,34092c37-960c-5ea3-9034-ff96334a7b40,CC-BY-4.0,HumanObservation,"{""airTemperature_C"":null,""annualPrecipitation_...",34092c37-960c-5ea3-9034-ff96334a7b40,NaN,present,"Kuhnt, A. (2014). Bemerkenswerte ""Schleimpilze...",2014,2014,...,Eumycetozoa.com,Protozoa,Mycetozoa,Myxomycetes,Physarales,Physaraceae,Willkommlangea,reticulata,species,"(Alb. & Schwein.) Kuntze, 1891"
34584,31bfaf62-a4d6-5483-8443-49bd73220598-ew,CC-BY-4.0,HumanObservation,"{""airTemperature_C"":3.8,""annualPrecipitation_m...",31bfaf62-a4d6-5483-8443-49bd73220598-ew,NaN,present,"Gmoshinskiy, V. I.; Matveev, A. V. (2016). Sea...",2016,2016,...,Eumycetozoa.com,Protozoa,Mycetozoa,Myxomycetes,Physarales,Physaraceae,Willkommlangea,reticulata,species,"(Alb. & Schwein.) Kuntze, 1891"
34585,8bc3402b-7064-555f-b885-33b82a932c28-aa,CC-BY-4.0,HumanObservation,NaN,8bc3402b-7064-555f-b885-33b82a932c28-aa,NaN,present,"Záhorovská, E. (1989). Príspevok k poznaniu sl...",1989-08-28,1989,...,Eumycetozoa.com,Protozoa,Mycetozoa,Myxomycetes,Physarales,Physaraceae,Willkommlangea,reticulata,species,"(Alb. & Schwein.) Kuntze, 1891"
34586,4e38fe04-ba6b-57d0-9675-63872aac2b2c-x,CC-BY-4.0,HumanObservation,"{""airTemperature_C"":null,""annualPrecipitation_...",4e38fe04-ba6b-57d0-9675-63872aac2b2c-x,NaN,present,"Gmoshinskiy, V. I.; Kireeva, N. I.; Sakulin, S...",2025-06-30,2025,...,Eumycetozoa.com,Protozoa,Mycetozoa,Myxomycetes,Physarales,Physaraceae,Willkommlangea,reticulata,species,"(Alb. & Schwein.) Kuntze, 1891"


In [58]:
safe_get_csv_details(csv_path) 

Trying separator=None
object of type 'NoneType' has no len()
Trying separator='\t'


/tmp/ipykernel_43186/3245544550.py:28: UserWarning: Could not read the CSV: csv='../_data/gbif_data_archive/working_copy/data/001-2fee34a0-a302-3c12-8916-bd915a1c0e50-download-data/occurrence.txt' -->
e=TypeError("object of type 'NoneType' has no len()")
  warnings.warn(f"Could not read the CSV: {csv=} -->\n{e=}")
/tmp/ipykernel_43186/3245544550.py:5: DtypeWarning: Columns (0: minimumElevationInMeters, 1: maximumElevationInMeters) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(csv, sep=separator, on_bad_lines='warn')


{'column_count': 38,
 'row_count': 34588,
 'head_size': 5,
 'raw_head': 'id\tlicense\tbasisOfRecord\tdynamicProperties\toccurrenceID\tgeoreferenceVerificationStatus\toccurrenceStatus\tassociatedReferences\teventDate\tyear\tmonth\tday\tcountry\tcountryCode\tstateProvince\tlocality\tverbatimLocality\tminimumElevationInMeters\tmaximumElevationInMeters\tdecimalLatitude\tdecimalLongitude\tgeodeticDatum\tcoordinateUncertaintyInMeters\tcoordinatePrecision\tgeoreferencedBy\tverbatimIdentification\tidentifiedBy\tscientificName\tnameAccordingTo\tkingdom\tphylum\tclass\torder\tfamily\tgenus\tspecificEpithet\ttaxonRank\tscientificNameAuthorship\n\n65440495-15f3-5e13-a4f5-5ab1eafe8f7e-a\tCC-BY-4.0\tHumanObservation\t{"airTemperature_C":null,"annualPrecipitation_mmYr":null,"habitatPressure":"Near-natural","forestTypeDetailed":"mountain Fagus-Carpinus forest","consolidatedForestCategory":null,"microhabitat":"bark of Pinus pallasiana (scorched trunk)","substrateCategory":"Corticolous substrates","pH":

In [59]:
safe_get_csv_details(
    "dwc_data/masked-doi-10.34894-72gq4c/budburst_event.csv"
)

Trying separator=None
object of type 'NoneType' has no len()
Trying separator='\t'


/tmp/ipykernel_43186/3245544550.py:28: UserWarning: Could not read the CSV: csv='dwc_data/masked-doi-10.34894-72gq4c/budburst_event.csv' -->
e=TypeError("object of type 'NoneType' has no len()")
  warnings.warn(f"Could not read the CSV: {csv=} -->\n{e=}")


{'column_count': 1,
 'row_count': 87901,
 'head_size': 5,
 'raw_head': '"eventID","parentEventID","samplingProtocol","sampleSizeValue","sampleSizeUnit","eventDate","year","month","day","country","countryCode","verbatimLocality","minimumElevationInMeters","maximumElevationInMeters","decimalLatitude","decimalLongitude","geodeticDatum","type","language","institutionID","institutionCode"\n\n"HV1988",NA,NA,NA,NA,"1988",1988,NA,NA,"Netherlands","NL","Hoge Veluwe",NA,NA,NA,NA,NA,"Event","en","https://ror.org/01g25jp36","NIOO"\n\n"HV1988_119","HV1988",NA,NA,NA,"1988-04-28",1988,4,28,"Netherlands","NL","Hoge Veluwe",NA,NA,NA,NA,NA,"Event","en","https://ror.org/01g25jp36","NIOO"\n\n"HV1988_119_100","HV1988_119","https://doi.org/10.1098/rspb.2000.1363",1,"tree","1988-04-28",1988,4,28,"Netherlands","NL","Hoge Veluwe",44.714127,44.714127,52.038898,5.854294,"EPSG:4326","Event","en","https://ror.org/01g25jp36","NIOO"\n\n"HV1988_119_101","HV1988_119","https://doi.org/10.1098/rspb.2000.1363",1,"tree","

In [63]:
import csv

def get_delimiter(file_path, bytes = 4096):
    sniffer = csv.Sniffer()
    data = open(file_path, "r").read(bytes)
    delimiter = sniffer.sniff(data, delimiters=[",", "\t"]).delimiter
    return delimiter

In [67]:
from csv import Sniffer


def detect_delim(file, num_rows=1):
    sniffer = Sniffer()
    with open(file,'r') as f:
        for row in range(num_rows):
            line = next(f).strip()
            sniffed = sniffer.sniff(line)
    
    return sniffed.delimiter

In [64]:
get_delimiter("dwc_data/masked-doi-10.34894-72gq4c/budburst_event.csv")

','

In [70]:
detect_delim("dwc_data/masked-doi-10.34894-72gq4c/budburst_event.csv", 1)

','

In [69]:
detect_delim(csv_path, 1)

'\t'

In [66]:
get_csv_details(csv_path, get_delimiter(csv_path))

/tmp/ipykernel_43186/3245544550.py:5: ParserWarning: Skipping line 12: expected 17 fields, saw 18
Skipping line 35: expected 17 fields, saw 19
Skipping line 75: expected 17 fields, saw 18
Skipping line 76: expected 17 fields, saw 18
Skipping line 98: expected 17 fields, saw 18
Skipping line 99: expected 17 fields, saw 18
Skipping line 100: expected 17 fields, saw 19
Skipping line 103: expected 17 fields, saw 18
Skipping line 107: expected 17 fields, saw 18
Skipping line 108: expected 17 fields, saw 18
Skipping line 109: expected 17 fields, saw 18
Skipping line 110: expected 17 fields, saw 18
Skipping line 111: expected 17 fields, saw 19
Skipping line 112: expected 17 fields, saw 19
Skipping line 130: expected 17 fields, saw 19
Skipping line 137: expected 17 fields, saw 18
Skipping line 145: expected 17 fields, saw 19
Skipping line 148: expected 17 fields, saw 18
Skipping line 149: expected 17 fields, saw 18
Skipping line 159: expected 17 fields, saw 18
Skipping line 189: expected 17 fi

{'column_count': 1,
 'row_count': 26191,
 'head_size': 5,
 'raw_head': 'id\tlicense\tbasisOfRecord\tdynamicProperties\toccurrenceID\tgeoreferenceVerificationStatus\toccurrenceStatus\tassociatedReferences\teventDate\tyear\tmonth\tday\tcountry\tcountryCode\tstateProvince\tlocality\tverbatimLocality\tminimumElevationInMeters\tmaximumElevationInMeters\tdecimalLatitude\tdecimalLongitude\tgeodeticDatum\tcoordinateUncertaintyInMeters\tcoordinatePrecision\tgeoreferencedBy\tverbatimIdentification\tidentifiedBy\tscientificName\tnameAccordingTo\tkingdom\tphylum\tclass\torder\tfamily\tgenus\tspecificEpithet\ttaxonRank\tscientificNameAuthorship\n\n65440495-15f3-5e13-a4f5-5ab1eafe8f7e-a\tCC-BY-4.0\tHumanObservation\t{"airTemperature_C":null,"annualPrecipitation_mmYr":null,"habitatPressure":"Near-natural","forestTypeDetailed":"mountain Fagus-Carpinus forest","consolidatedForestCategory":null,"microhabitat":"bark of Pinus pallasiana (scorched trunk)","substrateCategory":"Corticolous substrates","pH":n

In [77]:
import itertools as itx

def get_raw_file_head(
    path: pl.Path, head_size: int = 10, max_line_length: int | None = None
) -> str:
    with open(path) as fp:
        head_lines = itx.islice(fp, head_size)
        if max_line_length is not None:
            head_lines = [hl[:max_line_length] for hl in head_lines]
        raw_head = "\n".join(head_lines)
    return raw_head


def get_csv_details(csv: pl.Path, separator: str | None = None) -> CSVDetails:
    head_size: int = 5
    raw_head = get_raw_file_head(csv, head_size=head_size)
    df = pd.read_csv(csv, sep=separator, on_bad_lines="warn")
    details: CSVDetails = {
        "column_count": df.shape[1],
        "row_count": df.shape[0],
        "head_size": head_size,
        "raw_head": raw_head,
        "dataframe_head": df.head(head_size).to_csv(None, sep=",", index=False),
        "descriptive_stats": df.describe(include="all").to_csv(
            None, sep=",", index=False
        ),
    }
    del df
    return details


def detect_delim(file, num_rows=1):
    sniffer = Sniffer()
    with open(file, "r") as f:
        for row in range(num_rows):
            line = next(f).strip()
            sniffed = sniffer.sniff(line)

    return sniffed.delimiter


def safe_get_csv_details(csv: pl.Path) -> CSVDetails | None:
    try:
        separator = detect_delim(csv, num_rows=1)
        return get_csv_details(csv, separator)
    except Exception as e:
        # logger.error(e)
        # logger.warning(e)
        warnings.warn(f"Could not read the CSV: {csv=} -->\n{e=}")
        return None

In [75]:
safe_get_csv_details(csv_path)

/tmp/ipykernel_43186/3871318244.py:17: DtypeWarning: Columns (0: minimumElevationInMeters, 1: maximumElevationInMeters) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(csv, sep=separator, on_bad_lines="warn")


{'column_count': 38,
 'row_count': 34588,
 'head_size': 5,
 'raw_head': 'id\tlicense\tbasisOfRecord\tdynamicProperties\toccurrenceID\tgeoreferenceVerificationStatus\toccurrenceStatus\tassociatedReferences\teventDate\tyear\tmonth\tday\tcountry\tcountryCode\tstateProvince\tlocality\tverbatimLocality\tminimumElevationInMeters\tmaximumElevationInMeters\tdecimalLatitude\tdecimalLongitude\tgeodeticDatum\tcoordinateUncertaintyInMeters\tcoordinatePrecision\tgeoreferencedBy\tverbatimIdentification\tidentifiedBy\tscientificName\tnameAccordingTo\tkingdom\tphylum\tclass\torder\tfamily\tgenus\tspecificEpithet\ttaxonRank\tscientificNameAuthorship\n\n65440495-15f3-5e13-a4f5-5ab1eafe8f7e-a\tCC-BY-4.0\tHumanObservation\t{"airTemperature_C":null,"annualPrecipitation_mmYr":null,"habitatPressure":"Near-natural","forestTypeDetailed":"mountain Fagus-Carpinus forest","consolidatedForestCategory":null,"microhabitat":"bark of Pinus pallasiana (scorched trunk)","substrateCategory":"Corticolous substrates","pH":

In [79]:
safe_get_csv_details((
    "../_data/gbif_data_archive/working_copy/data/"
    + "023-ac2bcb25-c6ee-3355-93d0-b1470d70c32c-download-data/"
    + "occurrence.txt"
))

{'column_count': 47,
 'row_count': 3,
 'head_size': 5,
 'raw_head': 'id\tmodified\tlanguage\tlicense\tinstitutionID\tcollectionID\tdatasetID\tinstitutionCode\tcollectionCode\tdatasetName\tbasisOfRecord\toccurrenceID\tcatalogNumber\trecordedBy\trecordedByID\tindividualCount\tsex\tlifeStage\tassociatedOrganisms\teventDate\thabitat\tsamplingProtocol\tcontinent\tcountry\tstateProvince\tcounty\tlocality\tdecimalLatitude\tdecimalLongitude\tcoordinateUncertaintyInMeters\tidentifiedBy\tidentifiedByID\tdateIdentified\tidentificationReferences\tnameAccordingToID\tnamePublishedInID\tnameAccordingTo\tnamePublishedIn\tnamePublishedInYear\tkingdom\tphylum\tclass\torder\tfamily\tgenus\tspecificEpithet\tscientificNameAuthorship\n\nIMQC1005461\t2025-10-25\ten\thttps://creativecommons.org/publicdomain/zero/1.0/legalcode\thttps://scientific-collections.gbif.org/institution/bc356e35-e4e1-4365-86ee-aa324284560a\thttps://scientific-collections.gbif.org/collection/c2d42306-5d0d-4fde-9dbd-cb07a5449ac4\thttps: